# **Stock Market in Minecraft Notes**

In [ ]:
!latex --version

In [ ]:
from manim import *
import random

%load_ext autoreload
%autoreload 2

#### Quote Driven Market
- Market makers collect and publish prices they will buy and sell the traded asset for

- They set their sell prices higher than their buy prices to earn a profit in exchange for providing liquidity, taking on risk, and exposing themselves to traders with more information on the traded asset

- The **only** options available to traders are to buy and sell at one of the market maker's prices

#### Limit Order Books
- Much more flexible as **all** traders can post buy or sell orders

$$ order: x = (p_x,w_x,t_x) $$
$ t_x = $ time submitted

$ w_x = $ size of order in units

$ p_x = $ price

$$ sell: w_x > 0 $$
$$ buy: w_x < 0$$


In [ ]:
%%manim -qm -v WARNING OrderShowcase

class OrderShowcase(Scene):
    def construct(self):
        COLOR_MAP = {
            "sell": RED_B,
            "buy": GREEN_B
        }

        # EQUATIONS --------

        # order eq
        order_equation = MathTex(r"\text{order: } x = (p_x,w_x,t_x)")

        # order eq variables
        tx = MathTex(r"t_x = \text{time submitted}")
        wx = MathTex(r"w_x = \text{size of order}")
        px = MathTex(r"p_x = \text{price}")
        variable_explanation = VGroup(tx, wx, px).arrange(DOWN, aligned_edge=LEFT, buff=0.5)

        # sell and buy explanations
        sell = MathTex(r"\text{sell: } w_x > 0", tex_to_color_map=COLOR_MAP)
        buy = MathTex(r"\text{buy: } w_x < 0", tex_to_color_map=COLOR_MAP)
        sign_explanation = VGroup(sell, buy).arrange(DOWN, aligned_edge=LEFT, buff=0.5)

        # ANIMATION --------

        order_equation.next_to(variable_explanation, UP, buff=.5)

        # display order equation and its associated variables
        self.play(Write(order_equation))
        self.wait(.5)
        self.play(FadeIn(variable_explanation))
        self.wait(.8)

        group1 = VGroup(order_equation, variable_explanation)

        # move and scale the order equation and its associated variables to the left side of the screen
        self.play(group1.animate.scale(.9).to_edge(LEFT, buff=2.5))

        # move sign explanation to the right side of the screen, scale it down
        sign_explanation.to_edge(RIGHT, buff=2.5).scale(.9).shift(UP * 0.1)
        sign_explanation.align_to(group1, UP + DOWN)

        # display the sign explanation
        self.play(FadeIn(sign_explanation))
        self.wait(1.5)

        groupALL = VGroup(order_equation, variable_explanation, sign_explanation)

        self.play(FadeOut(groupALL))


**Lot Size:** $\sigma$
- smallest amount of asset traded within LOB
- all orders have size: $w_x \in \{\pm k \sigma \rvert k=1,2,...\}$
- if an increment $\epsilon$ exists then orders have size: $w_x \in \{\pm(\sigma + k \epsilon) \rvert k=1,2,...\}$

**Tick Size:** $\pi$
- smallest possible price interval 
- all orders must be specified to accuracy of $\pi$
- all orders have price: $p_x \in \{k\pi \rvert k=1,2,...\}$

In [ ]:
%%manim -qm -v WARNING ResolutionShowcase

class ResolutionShowcase(Scene):
    def construct(self):
        # EQUATIONS --------

        # lot size stuff
        lotHeading = MathTex(r"\text{Lot Size: } \sigma")
        lotSize = MathTex(r"w_x \in \{\pm k \sigma \rvert k=1,2,...\}")
        lotIncrementSize = MathTex(r"w_x \in \{\pm(\sigma + k \epsilon) \rvert k=1,2,...\}")
        lotGroup = VGroup(lotHeading, lotSize, lotIncrementSize)

        epsilonPart = lotIncrementSize.set_color_by_tex(r"\epsilon", YELLOW) 

        # tick size stuff
        tickHeading = MathTex(r"\text{Tick Size: } \pi")
        tickSize = MathTex(r"p_x \in \{k\pi \rvert k=1,2,...\}")
        tickGroup = VGroup(tickHeading, tickSize)

        # ANIMATION --------

        i = 0
        for element in (lotGroup):
            if i == 0:
                element.to_edge(UP, buff=2)
            else:
                element.next_to(lotGroup[i-1],DOWN, buff=0.5).align_to(lotGroup[i-1], UP + DOWN)
                
            self.play(Write(element))
            i += 1
            self.wait(.5)
        self.wait(2)
        self.play(FadeOut(lotGroup))

        i = 0
        for element in (tickGroup):
            if i == 0:
                element.to_edge(UP, buff=2)
            else:
                element.next_to(tickGroup[i-1],DOWN, buff=0.5).align_to(tickGroup[i-1], UP + DOWN)
                
            self.play(Write(element))
            i += 1
            self.wait(.5)
        self.wait(1)
        self.play(FadeOut(tickGroup))


        self.wait(2)

The LOB, $\zeta(t)$ can be considered the set of all active orders in a market at a time t
- evolution of this LOB can also be considered a cadlag process

For a limit order x, it holds that:
$$ x \in \zeta(t_x), x \notin \lim_{t' \to t_x} \zeta(t') $$

$\zeta(t)$ can be further partitioned into:

$B(t), w_x < 0$ for buy orders

$A(t), w_x > 0$ for sell orders


In [ ]:
%%manim -qm -v WARNING LOBShowcase

class LOBShowcase(Scene):
    COLOR_MAP = {
            r"A(t)": RED_B,
            r"B(t)": GREEN_B
        }

    def randomDate(self):
        month = random.randint(1, 12)
        day = random.randint(1, 28) 
        year = random.randint(2020, 2024)
        return f"{year}/{month:02d}/{day:02d}"

    def createOrderGroup(self, start, end):
        orderGroup = VGroup()
        for i in range(start, end):
            price = round(random.uniform(1, 100), 2)
            size = random.randint(-100, 100)
            time = self.randomDate()
            order = MathTex(rf"x_{{{i}}} = ({price}, {size}, {time})")
            orderGroup.add(order)
        return orderGroup

    def playOrderGroup(self, orderGroup, buff=1, alignment=LEFT, speed=.5):
        i = 0
        for element in (orderGroup):
            if i == 0:
                element.to_edge(UP, buff=buff).to_edge(alignment, buff=buff)
            else:
                element.next_to(orderGroup[i-1],DOWN, buff=0.5).align_to(orderGroup[i-1], UP + DOWN)
                
            self.play(FadeIn(element), run_time=speed)
            i += 1

    def construct(self):

        #EQUATIONS --------

        # lob
        lobHeading = MathTex(r"\text{Limit Order Book: }\zeta (t)", substrings_to_isolate=[r"\zeta (t)"])
        lobSubstring = lobHeading.get_part_by_tex(r"\zeta (t)").set_color_by_gradient(GRAY_A, PURPLE_B)

        group1 = self.createOrderGroup(1, 8)
        group2 = self.createOrderGroup(8, 15)
        allOrders = VGroup(group1, group2)

        bidSet = MathTex(r"B(t) = \{x \in \zeta (t) \rvert w_x < 0\}", tex_to_color_map=self.COLOR_MAP)

        askSet = MathTex(r"A(t) = \{x \in \zeta (t) \rvert w_x > 0\}", tex_to_color_map=self.COLOR_MAP)

        # ANIMATION --------
        self.playOrderGroup(group1, buff=.6, alignment=LEFT, speed=.2)
        self.playOrderGroup(group2, buff=.6, alignment=RIGHT, speed=.2)
        self.wait(1)
        self.play(ReplacementTransform(allOrders, lobHeading))
        self.wait(1.5)

        self.play(lobHeading.animate.to_edge(UP, buff=1.5))

        bidSet.next_to(lobHeading, DOWN, buff=.5).align_to(lobHeading, UP + DOWN)
        askSet.next_to(bidSet, DOWN, buff=.5).align_to(lobHeading, UP + DOWN)

        self.play(Write(bidSet), Write(askSet))
        self.wait(2.5)



**Bid Price:** 
- highest stated price among active buy orders at time t
$$ b(t) := \underset{x \in B(t)}{\max} p_x $$

**Ask Price:** 
- lowest stated price among active sell orders at time t
$$ a(t) := \underset{x \in A(t)}{\min} p_x $$

**Bid Ask Spread:** 
$$s(t) := a(t) - b(t) $$

**Mid Price:** 
$$m(t) := \frac{a(t) + b(t)}{2}$$


In [ ]:
%%manim -qm -v WARNING bidaskShowcase

class bidaskShowcase(Scene):
    def construct(self):
        COLOR_MAP = {
            r"a(t)": RED_B,
            r"b(t)": GREEN_B
        }

        #EQUATIONS --------
        bid = MathTex(r"b(t) := \underset{x \in B(t)}{\max} p_x", tex_to_color_map=COLOR_MAP)
        ask = MathTex(r"a(t) := \underset{x \in A(t)}{\min} p_x", tex_to_color_map=COLOR_MAP)

        spread = MathTex(r"s(t) := a(t) - b(t)", tex_to_color_map=COLOR_MAP)
        mid = MathTex(r"m(t) := \frac{a(t) + b(t)}{2}", tex_to_color_map=COLOR_MAP, substrings_to_isolate=["m(t)"])
        midSubString = mid.get_part_by_tex("m(t)").set_color_by_gradient(GREEN_D, BLUE_A)

        # ANIMATION --------
        bid.to_edge(LEFT, buff=2)
        ask.to_edge(RIGHT, buff=2)
        bidaskgroup = VGroup(bid, ask)

        self.play(Write(bidaskgroup))
        self.wait(2)

        self.play(ReplacementTransform(bidaskgroup, spread))
        self.wait(2)

        self.play(spread.animate.to_edge(UP, buff=1.5))
        mid.next_to(spread, DOWN, buff=1).align_to(spread, UP + DOWN)
        self.play(Write(mid))
        self.wait(3)


In [ ]:
%%manim -qm -v WARNING bidaskGraph

class bidaskGraph(Scene):
    def createLines(self, chart, x):
        x_vals = [ _ for _ in range(x)]
        ask_vals = [random.randint(96, 99) for _ in range(x)]
        bid_vals = [random.randint(91, 95) for _ in range(x)]

        ask_line = chart.plot_line_graph(x_vals, ask_vals, add_vertex_dots=False, line_color=RED_A)
        bid_line = chart.plot_line_graph(x_vals, bid_vals, add_vertex_dots=False, line_color=GREEN_A)
        allLines = VGroup(ask_line, bid_line)
        spreadLine = Line(chart.c2p(x_vals[1], ask_vals[1]), chart.c2p(x_vals[1], bid_vals[1]), color=PURPLE_B, stroke_width=2)

        return ask_line, bid_line, allLines, spreadLine

    def construct(self):
        chart = Axes(
            x_range=[0, 10, 1],
            y_range=[90, 100, 1],
            x_length=6,
            y_length=6,
        ).add_coordinates()
        chart.to_edge(UP, buff=.5)

        ask_line, bid_line, allLines, spreadLine = self.createLines(chart, 10)

        x_label = Text("Min past time t", font_size=24).next_to(chart, DOWN, buff=0.2)
        y_label = Text("Price", font_size=24).next_to(chart, LEFT, buff=0.2).rotate(90 * DEGREES)
        spread_label = Text("Spread", font_size=19).next_to(spreadLine, RIGHT, buff=0.02).rotate(90 * DEGREES)

        self.play(DrawBorderThenFill(chart, run_time=2), Write(x_label), Write(y_label), run_time=2)
        self.play(Write(allLines), run_time=4)
        self.play(Create(spreadLine), run_time=2)
        self.play(Write(spread_label), run_time=2)

        self.wait(4)

        self.play(FadeOut(allLines), FadeOut(spreadLine), FadeOut(chart), FadeOut(x_label), FadeOut(y_label), FadeOut(spread_label))



**Depth:**
- Bid Side Depth: $n^{b}(p,t) = \underset{\{x \in B(t) | p_x = p\}}\sum w_x$

- Ask Side Depth: $n^{a}(p,t) = \underset{\{x \in A(t) | p_x = p\}}\sum w_x$

**Depth Profiles:**
- Bid Depth Profile: set of all ordered pairs $(p, n^{b}(p,t))$ at time t
- Ask Depth Profile: set of all ordered pairs $(p, n^{a}(p,t))$ at time t


In [ ]:
%%manim -qm -v WARNING DepthShowcase

class DepthShowcase(Scene):
    def construct(self):
        COLOR_MAP = {
            r"b": GREEN_B,
            r"a": RED_B,
            r"B(t)": GREEN_B,
            r"A(t)": RED_B
        }
        
        # EQUATIONS --------
        bidLabel = MathTex(r"\text{Bid Side Depth: }")
        askLabel = MathTex(r"\text{Ask Side Depth: }")

        bidSetLabel = MathTex(r"\text{Bid Side Depth Profile: }")
        askSetLabel = MathTex(r"\text{Ask Side Depth Profile: }")

        bidSideDepth = MathTex(r"n^{b}(p,t) = \underset{\{x \in B(t) | p_x = p\}}\sum w_x" , tex_to_color_map=COLOR_MAP)
        askSideDepth = MathTex(r"n^{a}(p,t) = \underset{\{x \in A(t) | p_x = p\}}\sum w_x", tex_to_color_map=COLOR_MAP)

        bidSideProfile = MathTex(r"(p, n^{b}(p,t)) ", tex_to_color_map=COLOR_MAP)
        askSideProfile = MathTex(r"(p, n^{a}(p,t)) ", tex_to_color_map=COLOR_MAP)

        # ANIMATION --------
        bidLabel.to_edge(UP, buff=1)
        bidSideDepth.next_to(bidLabel, DOWN, buff=.5).align_to(bidLabel, LEFT + RIGHT)
        bidSetLabel.next_to(bidSideDepth, DOWN, buff=1.5).align_to(bidSideDepth, LEFT + RIGHT)
        bidSideProfile.next_to(bidSetLabel, DOWN, buff=1.5).align_to(bidSetLabel, LEFT + RIGHT)

        askLabel.to_edge(UP, buff=1)
        askSideDepth.next_to(askLabel, DOWN, buff=.5).align_to(askLabel, LEFT + RIGHT)
        askSetLabel.next_to(askSideDepth, DOWN, buff=1.5).align_to(askSideDepth, LEFT + RIGHT)
        askSideProfile.next_to(askSetLabel, DOWN, buff=1.5).align_to(askSetLabel, LEFT + RIGHT)

        self.play(Write(bidLabel))
        self.play(Write(bidSideDepth))
        self.wait(1)
        self.play(Write(bidSetLabel))
        self.play(Write(bidSideProfile))
        self.wait(3)
        self.play(FadeOut(bidSideDepth), FadeOut(bidSideProfile), FadeOut(bidLabel), FadeOut(bidSetLabel))

        self.play(Write(askLabel))
        self.play(Write(askSideDepth))
        self.wait(1)
        self.play(Write(askSetLabel))
        self.play(Write(askSideProfile))
        self.wait(3)
        self.play(FadeOut(askSideDepth), FadeOut(askSideProfile), FadeOut(askLabel), FadeOut(askSetLabel))



        

In [ ]:
%%manim -qm -v WARNING DepthGraph

class DepthGraph(Scene):
    def generateDepths(self):
        bid_depths = [-5,-1,-3,-2]
        ask_depths = [2,3,1,4]
        graph_points = [(85.5, bid_depths[0]), (86, bid_depths[1]), (86.5, bid_depths[2]), (87, bid_depths[3]), (87.5, 0), (88, ask_depths[0]), (88.5, ask_depths[1]), (89, ask_depths[2]), (89.5, ask_depths[3])]
        return graph_points

    def drawBars(self, chart):
        graph_points = self.generateDepths()
        bars = VGroup()
        width = 0.2

        for x, y in graph_points:
            if y == 0:
                continue
            color = GREEN_D if y < 0 else PURE_RED
            bar = Polygon(
                chart.c2p(x - width, 0), # draw the four corners
                chart.c2p(x + width, 0), 
                chart.c2p(x + width, y), 
                chart.c2p(x - width, y),
                stroke_width=2,
                color=color,
                fill_opacity=0.8
            )
            bars.add(bar)

        self.play(DrawBorderThenFill(bars, run_time=2))

    def construct(self):
        sigma = MathTex(r"\sigma = 1")
        epsilon = MathTex(r"\epsilon = 1")

        tick = MathTex(r"\pi = 0.5")

        resolutions = VGroup(sigma, epsilon, tick).arrange(DOWN, aligned_edge=LEFT + RIGHT, buff=0.5)

        chart = Axes(
            y_range=[-5, 5, 1],
            x_range=[85, 90, .5],
            x_length=7,
            y_length=7,
            x_axis_config={"numbers_to_exclude": [85, 85.0]}
        ).add_coordinates(font_size=28)
        chart.to_edge(UP, buff=.5).to_edge(LEFT, buff=1.5)

        x_label = Text("Price (USD)", font_size=22).next_to(chart, DOWN, buff=0.2)
        chart.get_x_axis_label(x_label)
        y_label = MathTex(r"\text{Depth Available }(\sigma)", font_size=31).rotate(90 * DEGREES).next_to(chart, LEFT, buff=0.2)

        # ANIMATION --------
        i = 0
        for element in (resolutions):
            if i == 0:
                self.play(Write(element))
                self.wait(.5)
                self.play(element.animate.to_edge(UP, buff=2))
            else:
                element.next_to(resolutions[i-1],DOWN, buff=0.5).align_to(resolutions[i-1], UP + DOWN)
                self.play(Write(element))
                self.wait(.5)
            i += 1

        self.play(resolutions.animate.to_edge(UP, buff=1).to_edge(RIGHT, buff = 1))

        self.play(DrawBorderThenFill(chart, run_time=2), Write(x_label), Write(y_label), run_time=2)
        self.drawBars(chart)
        self.wait(4)




**Relative price:** 
- From bid: 
$$\delta^{b}(p) := \lvert b(t)-p \rvert$$
- From ask: 
$$\delta^{a}(p) := \lvert a(t)-p \rvert$$

- I defined them in this way as this will be useful in our probability density function for incoming limit orders later 



**Relative Depth:**

- Bid Side Relative Depth:
$$ N^{b}(p,t) := \underset{\{x \in B(t)| \delta^{b}\}}\sum w_x $$

- Ask Side Relative Depth:
$$ N^{a}(p,t) := \underset{\{x \in B(t)| \delta^{a}\}}\sum w_x $$

**Relative Depth Profiles:**
- Bid Relative Depth Profile: set of all ordered pairs $(p, N^{b}(p,t))$ at time t
- Ask Relative Depth Profile: set of all ordered pairs $(p, N^{a}(p,t))$ at time t


**Order Arrivals:**
- LMT buy or sell orders arrive at a relative distance $\delta$ from the bid/ask at independent exponential times with ratio $\lambda(\delta)$
$$ \lambda(\delta) = \frac{k}{\delta^{\omega}} $$
k = constant that scales overall market volume

$\omega$ = exponent which dictates how fast probability drops off at further distance

- Market orders arrive at independent exponential times with rate $\mu$

- LMT cancellations occure at a relative distance $\delta$ from the bid/ask at a rate proportional to the number of orders at that distance; If the cancellation function is $\theta(\delta)$ and the number of orders at that price is z then the cancellation rate for that price level is $\theta(\delta)z$
